<a href="https://colab.research.google.com/github/Lthao-stack/Al-ve-suc-khoe-gioi-/blob/main/okeee.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
!pip install gradio scikit-learn pandas -q

import re
import gradio as gr
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

APP_NAME = "TRỢ LÝ ẢO HỖ TRỢ TRẢ LỜI CÁC CÂU HỎI THƯỜNG GẶP VỀ SỨC KHỎE GIỚI TÍNH"

def kb(intent, topic, keywords, answer, advice, seek_help):
    return {
        "intent": intent,
        "topic": topic,
        "keywords": keywords,
        "answer": answer,
        "advice": advice,
        "seek_help": seek_help
    }

knowledge_base = [
    kb("puberty", "dấu hiệu dậy thì ở nam",
       ["dậy thì nam", "nam dậy thì", "con trai dậy thì", "vỡ giọng", "mọc lông", "mộng tinh"],
       "Dậy thì ở nam thường có các thay đổi như cao nhanh, mọc lông, giọng trầm hơn, cơ thể có mùi hơn, da dễ nổi mụn và có thể xuất hiện mộng tinh.",
       "Giữ vệ sinh cá nhân, ngủ đủ, ăn uống đều, vận động hợp lý và không so sánh cơ thể mình với bạn bè.",
       "Nên hỏi bác sĩ nếu trên khoảng 14-15 tuổi vẫn chưa có dấu hiệu dậy thì rõ, hoặc có đau/sưng/bất thường kéo dài."),

    kb("puberty", "dậy thì ở nữ",
       ["dậy thì nữ", "con gái dậy thì", "phát triển ngực", "có kinh lần đầu", "kinh nguyệt đầu tiên"],
       "Dậy thì ở nữ thường có phát triển ngực, thay đổi vóc dáng, mọc lông, da dễ nổi mụn và xuất hiện kinh nguyệt.",
       "Chuẩn bị kiến thức về kinh nguyệt, giữ vệ sinh cơ thể và theo dõi các thay đổi của bản thân.",
       "Nên đi khám nếu dậy thì quá sớm, quá muộn, đau nhiều hoặc ra máu bất thường."),

    kb("puberty", "mụn tuổi dậy thì",
       ["mụn tuổi dậy thì", "dậy thì nổi mụn", "vì sao dậy thì nổi mụn"],
       "Tuổi dậy thì dễ nổi mụn do thay đổi hormone làm tuyến dầu hoạt động mạnh hơn.",
       "Rửa mặt nhẹ nhàng, không nặn mụn, ngủ đủ và tránh bôi sản phẩm không rõ nguồn gốc.",
       "Nên khám da liễu nếu mụn viêm nặng, đau nhiều, để lại sẹo hoặc kéo dài."),

    kb("menstruation", "trễ kinh",
       ["trễ kinh", "chậm kinh", "mất kinh", "chưa tới tháng", "chưa có kinh"],
       "Trễ kinh có thể do stress, thức khuya, thay đổi cân nặng, rối loạn sinh hoạt hoặc mang thai nếu có quan hệ nguy cơ.",
       "Theo dõi chu kỳ, ngủ đủ, giảm stress. Nếu có quan hệ nguy cơ, nên thử thai đúng thời điểm.",
       "Nên đi khám nếu trễ kinh kéo dài nhiều tuần, lặp lại nhiều chu kỳ hoặc kèm đau/ra máu bất thường."),

    kb("menstruation", "kinh nguyệt không đều",
       ["kinh nguyệt không đều", "kinh không đều", "rối loạn kinh nguyệt", "chu kỳ thay đổi"],
       "Kinh nguyệt không đều có thể gặp ở tuổi dậy thì, khi stress, thức khuya, thay đổi cân nặng hoặc nội tiết chưa ổn định.",
       "Ghi lại chu kỳ 2-3 tháng, ngủ đủ, ăn uống đều và giảm căng thẳng.",
       "Nên đi khám nếu mất kinh nhiều tháng, rong kinh, đau dữ dội hoặc ra máu bất thường."),

    kb("menstruation", "đau bụng kinh",
       ["đau bụng kinh", "đau ngày đèn đỏ", "đau khi tới tháng"],
       "Đau bụng kinh nhẹ đến vừa có thể gặp do tử cung co bóp. Đau quá nhiều hoặc ảnh hưởng sinh hoạt thì cần chú ý.",
       "Có thể nghỉ ngơi, chườm ấm bụng dưới, uống đủ nước và tránh thức khuya.",
       "Nên đi khám nếu đau dữ dội, ngất, sốt, ra máu quá nhiều hoặc đau khác thường."),

    kb("menstruation", "máu kinh màu nâu",
       ["máu kinh màu nâu", "kinh màu nâu", "máu nâu khi có kinh"],
       "Máu kinh màu nâu có thể gặp ở đầu hoặc cuối kỳ kinh do máu ra chậm hơn.",
       "Theo dõi lượng máu, số ngày hành kinh và triệu chứng kèm theo.",
       "Nên đi khám nếu máu nâu kéo dài, có mùi hôi, đau nhiều, ra máu giữa kỳ hoặc nghi ngờ mang thai."),

    kb("contraception", "bao cao su",
       ["bao cao su", "dùng bao", "dùng bao có an toàn không", "bao cao su tránh thai"],
       "Bao cao su giúp giảm nguy cơ mang thai ngoài ý muốn và giảm nguy cơ một số bệnh lây truyền qua đường tình dục nếu dùng đúng và nhất quán.",
       "Cần dùng từ đầu đến cuối theo hướng dẫn, kiểm tra bao còn hạn và không dùng lại.",
       "Nên hỏi nhân viên y tế nếu bao rách, tuột hoặc có tình huống nguy cơ."),

    kb("contraception", "bao cao su bị rách",
       ["rách bao", "tuột bao", "bao cao su bị rách", "bao bị tuột"],
       "Bao cao su bị rách hoặc tuột làm tăng nguy cơ mang thai ngoài ý muốn và bệnh lây truyền qua đường tình dục.",
       "Cần xác định có xuất tinh không, thời điểm xảy ra và có tiếp xúc trực tiếp không.",
       "Nên hỏi dược sĩ hoặc nhân viên y tế càng sớm càng tốt nếu vừa có tình huống nguy cơ."),

    kb("pregnancy_risk", "nguy cơ mang thai",
       ["có thai không", "mang thai không", "dính bầu", "quan hệ lần đầu", "cọ xát", "xuất tinh gần vùng kín"],
       "Có thể có nguy cơ mang thai nếu tinh dịch tiếp xúc trực tiếp gần hoặc trong vùng âm đạo. Chỉ cọ xát ngoài khi còn mặc quần áo thường có nguy cơ rất thấp.",
       "Bình tĩnh xác định tình huống cụ thể, theo dõi kỳ kinh và thử thai đúng thời điểm nếu có nguy cơ.",
       "Nên hỏi nhân viên y tế nếu có xuất tinh gần vùng kín, trễ kinh hoặc không chắc tình huống."),

    kb("contraception", "thuốc tránh thai khẩn cấp",
       ["thuốc tránh thai khẩn cấp", "uống thuốc khẩn cấp", "tránh thai khẩn cấp"],
       "Thuốc tránh thai khẩn cấp là biện pháp dự phòng sau tình huống có nguy cơ mang thai ngoài ý muốn, không phải biện pháp dùng thường xuyên.",
       "Chỉ cân nhắc khi có nguy cơ rõ và nên hỏi dược sĩ/nhân viên y tế để dùng đúng.",
       "Nên đi khám/hỏi dược sĩ nếu nôn nhiều, rối loạn kinh kéo dài hoặc không chắc cách dùng."),

    kb("gynecology", "ngứa vùng kín",
       ["ngứa vùng kín", "vùng kín bị ngứa", "ngứa dưới đó", "khó chịu vùng kín"],
       "Ngứa vùng kín có thể do kích ứng, vệ sinh chưa phù hợp, mặc đồ quá chật, nấm/viêm nhiễm hoặc STI nếu có yếu tố nguy cơ.",
       "Giữ vùng kín khô thoáng, tránh gãi, tránh thụt rửa sâu và không tự dùng thuốc mạnh.",
       "Nên đi khám nếu ngứa kéo dài, kèm khí hư mùi hôi, đau rát, tiểu buốt hoặc có quan hệ nguy cơ."),

    kb("gynecology", "khí hư bất thường",
       ["khí hư", "dịch âm đạo", "dịch vàng", "dịch xanh", "vón cục", "khí hư mùi hôi"],
       "Khí hư có thể tăng theo chu kỳ. Nhưng nếu đổi màu, vón cục, có mùi hôi hoặc kèm ngứa rát thì có thể gợi ý viêm nhiễm.",
       "Không tự đặt thuốc hoặc dùng dung dịch mạnh. Vệ sinh nhẹ nhàng và giữ khô thoáng.",
       "Nên đi khám nếu khí hư vàng/xanh/xám, vón cục, mùi hôi hoặc kèm ngứa rát."),

    kb("urinary", "tiểu buốt tiểu rắt",
       ["tiểu buốt", "tiểu rắt", "đau khi đi tiểu", "viêm đường tiểu", "tiểu ra máu"],
       "Tiểu buốt hoặc tiểu rắt có thể liên quan kích ứng hoặc viêm đường tiểu, đặc biệt nếu kèm đau, sốt hoặc tiểu ra máu.",
       "Uống đủ nước, không nhịn tiểu và không tự dùng kháng sinh.",
       "Nên đi khám nếu tiểu buốt kéo dài, sốt, đau bụng dưới, đau lưng hoặc tiểu ra máu."),

    kb("sti_std", "bệnh lây truyền qua đường tình dục",
       ["sti", "std", "bệnh lây", "quan hệ không an toàn", "mụn sinh dục", "vết loét", "hiv", "hpv", "lậu", "giang mai", "herpes"],
       "STI có thể gây tiểu buốt, dịch lạ, mùi hôi, nổi mụn/vết loét hoặc ngứa rát. Tuy nhiên nhiều STI có thể ít hoặc không có triệu chứng.",
       "Không nên tự đoán bệnh qua mạng. Nếu có nguy cơ, xét nghiệm và khám là cách đáng tin cậy hơn.",
       "Nên đi khám/xét nghiệm nếu có quan hệ nguy cơ hoặc triệu chứng bất thường."),

    kb("sti_std", "HIV",
       ["hiv", "xét nghiệm hiv", "test hiv", "hiv lây qua đường nào"],
       "HIV có thể lây qua máu, quan hệ tình dục không an toàn và từ mẹ sang con. HIV không lây qua tiếp xúc thông thường như bắt tay, ăn chung hoặc dùng chung nhà vệ sinh.",
       "Phòng tránh bằng quan hệ an toàn, không dùng chung kim tiêm và xét nghiệm khi có nguy cơ.",
       "Nên xét nghiệm nếu có quan hệ nguy cơ, tiếp xúc máu nguy cơ hoặc lo lắng sau tình huống cụ thể."),

    kb("sti_std", "HPV",
       ["hpv", "virus hpv", "sùi mào gà"],
       "HPV là một nhóm virus lây truyền chủ yếu qua tiếp xúc tình dục. Một số type HPV có thể gây mụn cóc sinh dục, một số type liên quan đến ung thư cổ tử cung.",
       "Có thể phòng ngừa bằng vaccine HPV theo tư vấn y tế và quan hệ an toàn.",
       "Nên đi khám nếu có mụn/vết bất thường vùng sinh dục hoặc muốn tư vấn tiêm vaccine."),

    kb("lgbtq", "LGBTQ",
       ["lgbt", "lgbtq", "gay", "lesbian", "bisexual", "đồng tính", "song tính", "chuyển giới", "xu hướng tính dục"],
       "LGBTQ+ là thuật ngữ chỉ các nhóm đa dạng về xu hướng tính dục và bản dạng giới. LGBTQ+ không phải bệnh.",
       "Hãy tiếp cận chủ đề này bằng sự tôn trọng, không phán xét và không ép bản thân phải gắn nhãn ngay.",
       "Nên tìm người đáng tin cậy hoặc chuyên gia tâm lý nếu bạn bị kỳ thị, áp lực hoặc hoang mang kéo dài."),

    kb("lgbtq", "come out",
       ["come out", "chia sẻ xu hướng tính dục", "nói với gia đình", "có nên come out"],
       "Chia sẻ xu hướng tính dục với gia đình là quyết định cá nhân. Điều quan trọng nhất là sự an toàn và sự sẵn sàng của bạn.",
       "Chọn thời điểm phù hợp, chuẩn bị tâm lý và có người đáng tin cậy hỗ trợ nếu cần.",
       "Không nên tự ép mình chia sẻ nếu môi trường chưa an toàn."),

    kb("consent_safety", "đồng thuận",
       ["đồng thuận", "consent", "không muốn", "bị ép", "ép buộc", "quấy rối"],
       "Đồng thuận là khi cả hai tự nguyện, tỉnh táo, hiểu rõ và có quyền dừng lại bất cứ lúc nào. Im lặng hoặc bị ép không phải là đồng thuận.",
       "Hãy tôn trọng ranh giới của bản thân và người khác. Bạn có quyền nói không với điều khiến mình không thoải mái.",
       "Cần tìm hỗ trợ nếu có ép buộc, đe dọa, quấy rối hoặc xâm hại."),

    kb("consent_safety", "an toàn mạng xã hội",
       ["gửi ảnh", "ảnh riêng tư", "ảnh nhạy cảm", "bị đe dọa", "tống tiền", "người quen qua mạng"],
       "Chia sẻ hình ảnh riêng tư có thể gây rủi ro bị lưu lại, phát tán, đe dọa hoặc ép buộc.",
       "Không gửi thông tin/hình ảnh nhạy cảm. Chặn/báo cáo người quấy rối và tìm người lớn đáng tin cậy hỗ trợ.",
       "Cần báo người lớn/cơ quan hỗ trợ nếu bị đe dọa phát tán ảnh hoặc bị ép buộc."),

    kb("hygiene_body", "vệ sinh vùng kín",
       ["vệ sinh vùng kín", "rửa vùng kín", "dung dịch vệ sinh", "thụt rửa", "xà phòng vùng kín"],
       "Vệ sinh vùng kín đúng cách là rửa nhẹ nhàng bên ngoài, giữ khô thoáng, không thụt rửa sâu và không dùng sản phẩm quá mạnh.",
       "Thay đồ lót sạch, chọn đồ thoáng, lau khô nhẹ nhàng sau khi vệ sinh.",
       "Nên đi khám nếu có ngứa, mùi hôi, khí hư bất thường hoặc đau kéo dài."),

    kb("sexual_function", "trục trặc chức năng tình dục",
       ["xuất tinh sớm", "rối loạn cương", "khô rát", "đau khi quan hệ", "giảm ham muốn"],
       "Một số trục trặc chức năng tình dục có thể liên quan căng thẳng, tâm lý, lối sống hoặc vấn đề sức khỏe nền.",
       "Không tự dùng thuốc không rõ nguồn. Nên trao đổi với chuyên gia y tế nếu tình trạng kéo dài hoặc ảnh hưởng tâm lý.",
       "Nên đi khám nếu đau, chảy máu, rối loạn kéo dài hoặc ảnh hưởng nhiều đến sinh hoạt.")
]

df = pd.DataFrame(knowledge_base)

documents = (
    df["intent"].astype(str) + " " +
    df["topic"].astype(str) + " " +
    df["keywords"].apply(lambda x: " ".join(x)) + " " +
    df["answer"].astype(str) + " " +
    df["advice"].astype(str)
).tolist()

vectorizer = TfidfVectorizer(
    lowercase=True,
    ngram_range=(1, 3),
    max_features=25000,
    sublinear_tf=True
)
tfidf_matrix = vectorizer.fit_transform(documents)

NORMALIZATION_MAP = {
    "ko": "không",
    "k ": "không ",
    "khong": "không",
    "qhe": "quan hệ",
    "qh": "quan hệ",
    "bcs": "bao cao su",
    "ngày dâu": "kinh nguyệt",
    "đèn đỏ": "kinh nguyệt",
    "tới tháng": "kinh nguyệt",
    "đến tháng": "kinh nguyệt",
    "chưa tới tháng": "trễ kinh",
    "ra dịch": "khí hư",
    "dịch có mùi": "khí hư mùi hôi",
    "đi tiểu rát": "tiểu buốt",
    "đi tiểu đau": "tiểu buốt",
    "nổi hột": "mụn",
    "dính bầu": "mang thai",
    "tự sướng": "thủ dâm",
    "không muốn mà bắt": "ép buộc"
}

INTENT_KEYWORDS = {
    "menstruation": [
        "kinh nguyệt", "kinh không đều", "rối loạn kinh nguyệt", "trễ kinh",
        "chậm kinh", "mất kinh", "rong kinh", "đau bụng kinh", "máu kinh"
    ],
    "pregnancy_risk": [
        "có thai", "mang thai", "dính bầu", "cọ xát", "xuất tinh",
        "trễ kinh sau quan hệ", "quan hệ lần đầu"
    ],
    "contraception": [
        "bao cao su", "thuốc tránh thai", "rách bao", "tuột bao",
        "tránh thai khẩn cấp", "dùng bao"
    ],
    "gynecology": [
        "khí hư", "dịch âm đạo", "dịch trắng", "dịch vàng", "dịch xanh",
        "vón cục", "mùi hôi", "viêm âm đạo", "ngứa vùng kín"
    ],
    "urinary": [
        "tiểu buốt", "tiểu rắt", "đau khi đi tiểu", "viêm đường tiểu", "tiểu ra máu"
    ],
    "sti_std": [
        "ngứa", "rát", "mụn", "vết loét", "hiv", "hpv", "lậu",
        "giang mai", "herpes", "sùi mào gà", "sti", "std", "bệnh lây",
        "quan hệ không an toàn"
    ],
    "puberty": [
        "dậy thì", "vỡ giọng", "mộng tinh", "mọc lông", "chưa có kinh",
        "thủ dâm", "mùi cơ thể"
    ],
    "hygiene_body": [
        "vệ sinh", "dung dịch vệ sinh", "thụt rửa", "vùng kín",
        "xà phòng", "đồ lót"
    ],
    "lgbtq": [
        "gay", "lesbian", "bisexual", "lgbt", "lgbtq", "đồng tính",
        "song tính", "come out", "chuyển giới", "xu hướng tính dục"
    ],
    "consent_safety": [
        "ép", "ép buộc", "không muốn", "gửi ảnh", "ảnh nhạy cảm",
        "xâm hại", "lạm dụng", "đồng thuận", "đe dọa", "quấy rối",
        "người quen qua mạng"
    ],
    "sexual_function": [
        "xuất tinh sớm", "rối loạn cương", "khô rát", "đau khi quan hệ", "ham muốn"
    ]
}

DOMAIN_KEYWORDS = sorted(set(sum(INTENT_KEYWORDS.values(), [])))

OUT_OF_SCOPE_KEYWORDS = [
    "thời tiết", "điện thoại", "laptop", "game", "nấu", "cháo", "bóng đá",
    "toán", "code", "python", "palworld", "liên quân", "giảm cân", "visa", "ngân hàng"
]

class MemoryAgent:
    def __init__(self):
        self.state = {"last_intent": None, "context": {}, "asked": set()}

    def update(self, intent=None, context=None):
        if intent:
            self.state["last_intent"] = intent
        if context:
            for k, v in context.items():
                if v not in [None, False, [], ""]:
                    self.state["context"][k] = v

    def merge_context(self, context):
        merged = dict(self.state["context"])
        for k, v in context.items():
            if v not in [None, False, [], ""]:
                merged[k] = v
        return merged

    def reset(self):
        self.state = {"last_intent": None, "context": {}, "asked": set()}

    def already_asked(self, key):
        if key in self.state["asked"]:
            return True
        self.state["asked"].add(key)
        return False

memory_agent = MemoryAgent()

class InputAnalysisAgent:
    def normalize(self, text):
        text = (text or "").lower().strip()
        for old, new in NORMALIZATION_MAP.items():
            text = text.replace(old, new)
        return text

    def run(self, message):
        text = self.normalize(message)

        if text in ["chào", "xin chào", "hello", "hi", "bot ơi", "bạn ơi", "alo"]:
            return {"text": text, "intent": "greeting"}

        if any(x in text for x in ["cảm ơn", "tạm biệt", "bye", "kết thúc", "xong rồi"]):
            return {"text": text, "intent": "ending"}

        if any(x in text for x in OUT_OF_SCOPE_KEYWORDS) and not any(x in text for x in DOMAIN_KEYWORDS):
            return {"text": text, "intent": "out_of_scope"}

        best_intent = None
        best_hit = 0

        for item in knowledge_base:
            hit = 0
            for kw in item["keywords"]:
                if kw in text:
                    hit += len(kw.split())
            if hit > best_hit:
                best_hit = hit
                best_intent = item["intent"]

        if best_intent and best_hit >= 2:
            return {"text": text, "intent": best_intent}

        scores = {}
        for intent, kws in INTENT_KEYWORDS.items():
            scores[intent] = sum(1 for kw in kws if kw in text)

        best_intent = max(scores, key=scores.get)

        if scores[best_intent] == 0:
            if len(text.split()) <= 5:
                return {"text": text, "intent": "missing_info"}
            return {"text": text, "intent": "out_of_scope"}

        return {"text": text, "intent": best_intent}

class ContextExtractionAgent:
    def run(self, text):
        context = {
            "age": None,
            "gender": None,
            "symptoms": [],
            "duration": None,
            "has_sexual_risk": False,
            "used_condom": None,
            "period_late": False,
            "stress": False,
            "fear": False,
            "coercion": False,
            "cycle_change": False
        }

        age = re.search(r"(\d{1,2})\s*tuổi", text)
        if age:
            context["age"] = age.group(1)

        if any(x in text for x in ["nam", "con trai"]):
            context["gender"] = "nam"
        if any(x in text for x in ["nữ", "con gái"]):
            context["gender"] = "nữ"

        symptoms = [
            "ngứa", "rát", "đau", "mụn", "loét", "khí hư", "mùi hôi",
            "tiểu buốt", "tiểu rắt", "trễ kinh", "mộng tinh", "vỡ giọng",
            "khô rát", "rong kinh", "đau bụng kinh", "mất kinh",
            "dịch vàng", "dịch xanh", "vón cục", "kinh không đều"
        ]
        context["symptoms"] = [s for s in symptoms if s in text]

        duration = re.search(r"(\d+)\s*(ngày|tuần|tháng|năm)", text)
        if duration:
            context["duration"] = duration.group(0)

        if any(x in text for x in ["quan hệ", "cọ xát", "xuất tinh", "không dùng bao", "rách bao", "tuột bao"]):
            context["has_sexual_risk"] = True

        if "không dùng bao" in text:
            context["used_condom"] = False
        elif "bao cao su" in text or "dùng bao" in text:
            context["used_condom"] = True

        if any(x in text for x in ["trễ kinh", "chậm kinh", "mất kinh"]):
            context["period_late"] = True

        if any(x in text for x in ["stress", "căng thẳng", "thức khuya", "áp lực", "lo lắng"]):
            context["stress"] = True

        if any(x in text for x in ["lo quá", "sợ", "hoang mang", "ngại", "xấu hổ"]):
            context["fear"] = True

        if any(x in text for x in ["ép", "ép buộc", "không muốn", "đe dọa", "xâm hại", "lạm dụng", "quấy rối"]):
            context["coercion"] = True

        if any(x in text for x in ["kinh đến sớm", "kinh đến muộn", "kinh tới sớm", "kinh tới muộn", "kinh không đều", "chu kỳ thay đổi"]):
            context["cycle_change"] = True

        return context

class SafetyTriageAgent:
    def run(self, intent, context, text):
        red_flags = []

        if context.get("coercion"):
            red_flags.append("Có dấu hiệu ép buộc hoặc không an toàn.")

        if any(x in text for x in ["chảy máu nhiều", "đau dữ dội", "ngất", "sốt cao", "mệt lả", "tiểu ra máu"]):
            red_flags.append("Có dấu hiệu cần được hỗ trợ y tế sớm.")

        if red_flags:
            return {"level": "🔴 CẦN HỖ TRỢ SỚM", "red_flags": red_flags}

        if intent in ["sti_std", "gynecology", "urinary", "consent_safety"]:
            return {"level": "🟠 CẦN THEO DÕI/KHÁM NẾU KÉO DÀI", "red_flags": []}

        if intent in ["pregnancy_risk", "contraception", "menstruation"]:
            return {"level": "🟡 CẦN THEO DÕI THÊM", "red_flags": []}

        return {"level": "🟢 CHƯA THẤY DẤU HIỆU KHẨN CẤP", "red_flags": []}

class MissingInfoAgent:
    def run(self, intent, context, text):
        questions = []

        if intent == "pregnancy_risk":
            if not context.get("has_sexual_risk"):
                questions.append("Tình huống là cọ xát ngoài, có quan hệ hay có xuất tinh gần vùng kín không?")
            if not context.get("duration"):
                questions.append("Việc đó xảy ra cách đây bao lâu?")
            if not context.get("period_late"):
                questions.append("Hiện có trễ kinh hoặc gần đến ngày kinh chưa?")

        elif intent in ["sti_std", "gynecology", "urinary"]:
            if not context.get("symptoms"):
                questions.append("Triệu chứng cụ thể là gì: ngứa, rát, đau, nổi mụn, dịch lạ hay tiểu buốt?")
            if not context.get("duration"):
                questions.append("Triệu chứng xuất hiện bao lâu rồi?")
            if intent == "sti_std" and not context.get("has_sexual_risk"):
                questions.append("Gần đây có quan hệ nguy cơ hoặc không dùng bao không?")

        elif intent == "puberty":
            if not context.get("age"):
                questions.append("Bạn bao nhiêu tuổi?")
            if not context.get("gender"):
                questions.append("Bạn là nam hay nữ?")
            if not context.get("symptoms"):
                questions.append("Biểu hiện cụ thể bạn đang lo là gì?")

        questions = questions[:2]

        if questions:
            key = intent + "|" + "|".join(questions)
            if not memory_agent.already_asked(key):
                return "Mình cần thêm một chút thông tin để tư vấn đúng hơn:\n\n" + "\n".join(f"- {q}" for q in questions)

        return None

class KnowledgeRetrievalAgent:
    def run(self, text, intent):
        candidate_indexes = df[df["intent"] == intent].index.tolist()

        best_idx = None
        best_hit = 0

        for i in candidate_indexes:
            item = df.iloc[i]
            hit = 0
            for kw in item["keywords"]:
                if kw in text:
                    hit += len(kw.split())
            if hit > best_hit:
                best_hit = hit
                best_idx = i

        if best_idx is not None and best_hit >= 2:
            row = df.iloc[best_idx].to_dict()
            row["score"] = 1.0
            return row

        query = f"{intent} {text}"
        user_vec = vectorizer.transform([query])
        scores = cosine_similarity(user_vec, tfidf_matrix).flatten()

        if candidate_indexes:
            best_idx = max(candidate_indexes, key=lambda i: scores[i])
        else:
            best_idx = int(scores.argmax())

        best_score = float(scores[best_idx])
        threshold = 0.13 if intent in ["menstruation", "pregnancy_risk", "puberty", "hygiene_body"] else 0.16

        if best_score < threshold:
            return None

        row = df.iloc[best_idx].to_dict()
        row["score"] = best_score
        return row

class AdvicePlanningAgent:
    def run(self, intent, context, knowledge, safety, text):
        if knowledge is None:
            return None

        advice = knowledge["advice"]

        if intent == "menstruation":
            if context.get("cycle_change"):
                advice += "\n- Chu kỳ đến sớm hoặc muộn đôi khi chỉ là dao động tạm thời, nhất là khi stress hoặc thức khuya."
            if "đau bụng kinh" in context.get("symptoms", []):
                advice += "\n- Có thể chườm ấm bụng dưới và nghỉ ngơi; nếu đau dữ dội thì không nên cố chịu."
            if "rong kinh" in context.get("symptoms", []):
                advice += "\n- Hãy theo dõi lượng máu, số ngày ra máu và dấu hiệu chóng mặt/mệt lả."
            if context.get("stress"):
                advice += "\n- Stress và thức khuya có thể làm chu kỳ kinh bị lệch."

        if intent in ["gynecology", "sti_std"]:
            advice += "\n- Không thụt rửa sâu, không tự đặt thuốc hoặc bôi thuốc mạnh khi chưa rõ nguyên nhân."

        if intent == "urinary":
            advice += "\n- Uống đủ nước, không nhịn tiểu và không tự dùng kháng sinh."

        if intent == "consent_safety":
            advice += "\n- Nếu có tin nhắn đe dọa hoặc ép buộc, hãy lưu lại bằng chứng và tìm người đáng tin cậy hỗ trợ."

        return advice

class ResponseGenerationAgent:
    def empathy(self, context):
        if context.get("coercion"):
            return "Trước hết, bạn không có lỗi nếu đang bị ép làm điều mình không muốn."
        if context.get("fear"):
            return "Mình hiểu là bạn đang lo lắng, mình sẽ giải thích theo hướng dễ hiểu và an toàn nhé."
        return "Mình sẽ dựa trên thông tin bạn đưa để nhận định ban đầu nhé."

    def run(self, intent, context, safety, knowledge, advice):
        if knowledge is None:
            return (
                "Mình chưa tìm thấy tri thức đủ khớp trong cơ sở dữ liệu hiện tại nên chưa nên kết luận.\n\n"
                "Bạn có thể mô tả rõ hơn tình huống, thời gian xảy ra và triệu chứng chính không?"
            )

        red_flag_text = ""
        if safety["red_flags"]:
            red_flag_text = "\n".join(f"- {x}" for x in safety["red_flags"]) + "\n\n"

        return f"""
{self.empathy(context)}

Nhận định ban đầu:
{knowledge["answer"]}

Vì sao có thể như vậy:
Câu hỏi của bạn thuộc nhóm "{intent}". Hệ thống chỉ đưa ra nhận định ban đầu dựa trên thông tin bạn cung cấp, không chẩn đoán chắc chắn.

Mức độ cần lưu ý:
{safety["level"]}

{red_flag_text}Bạn nên làm gì:
{advice}

Khi nào cần đi khám/tìm hỗ trợ:
{knowledge["seek_help"]}

Lưu ý an toàn:
Thông tin này chỉ mang tính tham khảo, không thay thế tư vấn trực tiếp từ bác sĩ, dược sĩ hoặc chuyên gia phù hợp.
""".strip()

input_agent = InputAnalysisAgent()
context_agent = ContextExtractionAgent()
safety_agent = SafetyTriageAgent()
missing_agent = MissingInfoAgent()
retrieval_agent = KnowledgeRetrievalAgent()
advice_agent = AdvicePlanningAgent()
response_agent = ResponseGenerationAgent()

def chatbot(message, history):
    if not message or not message.strip():
        return "Bạn hãy nhập câu hỏi để mình hỗ trợ nhé."

    analysis = input_agent.run(message)
    text = analysis["text"]
    intent = analysis["intent"]

    if intent == "greeting":
        return (
            "Xin chào 👋\n\n"
            "Mình là trợ lý ảo AI hỗ trợ trả lời các câu hỏi thường gặp về sức khỏe giới tính.\n"
            "Bạn có thể mô tả tình huống của mình, mình sẽ trả lời theo hướng dễ hiểu, an toàn và không phán xét."
        )

    if intent == "ending":
        memory_agent.reset()
        return "Cuộc trò chuyện đã được kết thúc. Mình đã làm mới ngữ cảnh cho câu hỏi tiếp theo."

    if intent == "out_of_scope":
        return (
            "Mình chỉ hỗ trợ các câu hỏi về sức khỏe giới tính, sức khỏe sinh sản, dậy thì, tránh thai, "
            "bệnh lây truyền qua đường tình dục, kinh nguyệt, vệ sinh vùng kín, đồng thuận và các vấn đề liên quan.\n\n"
            "Bạn hãy hỏi lại đúng phạm vi để mình hỗ trợ chính xác hơn nhé."
        )

    if intent == "missing_info":
        return (
            "Mình chưa đủ thông tin để nhận định.\n\n"
            "Bạn có thể nói rõ hơn: vấn đề đang gặp là gì, xảy ra bao lâu rồi, có triệu chứng nào kèm theo và điều bạn lo nhất là gì?"
        )

    context = context_agent.run(text)
    context = memory_agent.merge_context(context)
    memory_agent.update(intent=intent, context=context)

    safety = safety_agent.run(intent, context, text)

    followup = missing_agent.run(intent, context, text)
    if followup:
        return followup

    knowledge = retrieval_agent.run(text, intent)
    advice = advice_agent.run(intent, context, knowledge, safety, text)

    return response_agent.run(intent, context, safety, knowledge, advice)

custom_css = """
.gradio-container {
    background: #0f172a !important;
    color: white !important;
}

#main-title {
    text-align: center;
    padding: 18px;
    border-radius: 18px;
    background: linear-gradient(135deg, #1e293b, #334155);
    margin-bottom: 18px;
}

#side-panel {
    background: #111827;
    border: 1px solid #334155;
    border-radius: 18px;
    padding: 16px;
}

textarea {
    border-radius: 16px !important;
}

footer {
    display: none !important;
}
"""

with gr.Blocks(css=custom_css) as demo:
    gr.Markdown(
        f"""
<div id="main-title">

# {APP_NAME}

Hỗ trợ trả lời các câu hỏi thường gặp về sức khỏe giới tính theo hướng dễ hiểu, an toàn và không phán xét.

</div>
""",
    )

    with gr.Row():
        with gr.Column(scale=3):
            gr.ChatInterface(
                fn=chatbot,
                examples=[],
                textbox=gr.Textbox(
                    placeholder="Nhập câu hỏi của bạn tại đây...",
                    container=True,
                    scale=7
                )
            )

        with gr.Column(scale=1):
            gr.Markdown(
                """
<div id="side-panel">

### Phạm vi hỗ trợ

- Dậy thì nam/nữ
- Kinh nguyệt, trễ kinh
- Tránh thai, bao cao su
- Viêm nhiễm vùng kín
- STI/STD
- LGBTQ+
- Đồng thuận và an toàn cá nhân
- Vệ sinh vùng nhạy cảm

### Lưu ý

Thông tin chỉ mang tính tham khảo, không thay thế tư vấn y tế trực tiếp.

</div>
""",
            )

demo.launch(share=True)

/tmp/ipykernel_5246/319110143.py:647: DeprecationWarning: The 'css' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'css' to Blocks.launch() instead.
  with gr.Blocks(css=custom_css) as demo:
/usr/local/lib/python3.12/dist-packages/gradio/chat_interface.py:347: UserWarning: The 'tuples' format for chatbot messages is deprecated and will be removed in a future version of Gradio. Please set type='messages' instead, which uses openai-style 'role' and 'content' keys.
  self.chatbot = Chatbot(


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://7f110781daf5c1d5d7.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
